In [ ]:
import time
import os
import pandas as pd

class CustomVendingMachine:
    def __init__(self):
        # State Constants
        self.IDLE, self.ADD_MONEY, self.DISPENSE, self.REFUND = "IDLE", "ADD_MONEY", "DISPENSE", "REFUND"

        # Registers (Latching on Clock Edges)
        self.PS = self.IDLE
        self.accumulated_balance = 0
        self.latched_target_price = 255

        # Outputs (Combinational wires based on PS)
        self.dispense = 0
        self.return_change = 0

        # Dictionaries will be populated dynamically from Excel
        self.item_names = {0: "None"}
        self.prices = {0: 255}

    def evaluate_combinational_outputs(self, money_in, current_item_price):
        """Calculates combinational outputs and next-state routes instantly (always @(*))"""
        if self.PS == self.ADD_MONEY:
            is_enough_money = (money_in >= current_item_price)
            active_price = current_item_price
        else:
            is_enough_money = (self.accumulated_balance >= self.latched_target_price)
            active_price = self.latched_target_price

        # --- Output Mappings ---
        if self.PS in [self.IDLE, self.ADD_MONEY]:
            dispense_wire = 0
            change_wire = 0
        elif self.PS == self.DISPENSE:
            dispense_wire = 1
            change_wire = 0
        elif self.PS == self.REFUND:
            dispense_wire = 0
            if (self.accumulated_balance > active_price) or (not is_enough_money):
                change_wire = 1
            else:
                change_wire = 0
        else:
            dispense_wire, change_wire = 0, 0

        # --- Next State Calculations ---
        if self.PS == self.IDLE:
            ns = self.ADD_MONEY
        elif self.PS == self.ADD_MONEY:
            if money_in > 0:
                ns = self.DISPENSE if is_enough_money else self.REFUND
            else:
                ns = self.ADD_MONEY
        elif self.PS == self.DISPENSE:
            ns = self.REFUND
        else:
            ns = self.IDLE

        return ns, dispense_wire, change_wire

    def tick_clock_edge(self, ns, money_in, current_item_price):
        """Updates state variables and tracking values on the active clock edge (always @(posedge clk))"""
        if self.PS == self.IDLE:
            self.accumulated_balance = 0
            self.latched_target_price = 255
        elif self.PS == self.ADD_MONEY:
            if money_in > 0:
                self.accumulated_balance = money_in
                self.latched_target_price = current_item_price
        elif self.PS == self.REFUND:
            self.accumulated_balance = 0
            self.latched_target_price = 255

        self.PS = ns


# ==========================================
# EXCEL CONFIGURATION & GENERATOR
# ==========================================
excel_filename = "vending_items.xlsx"

# Standard columns expected in your spreadsheet:
# "Item Code", "Item Name", "Price"
if not os.path.exists(excel_filename):
    print(f"[*] '{excel_filename}' not found. Generating a mock dataset with 100 items for testing...")
    mock_data = {
        "Item Code": list(range(1, 101)),
        "Item Name": [f"Item Pack #{i}" for i in range(1, 101)],
        "Price": [10 + (i % 10) * 5 for i in range(1, 101)] # Dynamic pricing array (10, 15, 20...55)
    }
    df_mock = pd.DataFrame(mock_data)
    df_mock.to_excel(excel_filename, index=False)
    print(f"[+] Successfully generated mock inventory: '{excel_filename}'")

# Load configuration into your app maps using pandas dataframes
try:
    df = pd.read_excel(excel_filename)

    # Instantiate the system engine
    vm = CustomVendingMachine()

    # Populating system mapping lists directly from parsed columns
    for _, row in df.iterrows():
        code = int(row["Item Code"])
        name = str(row["Item Name"])
        price = int(row["Price"])

        vm.item_names[code] = f"{name} ({price})"
        vm.prices[code] = price

    print(f"[+] Loaded {len(vm.prices)-1} unique hardware lookup values from the spreadsheet database.\n")
except Exception as e:
    print(f"[ERROR] Failed parsing the excel file mapping data structural rules: {e}")
    exit(1)


# ==========================================
# RUNTIME APPLICATION LOOP
# ==========================================
user_item, user_money = 0, 0
print("="*55 + "\n   100+ EXCEL DATABASE PIPELINE RUNTIME ACTIVATED\n" + "="*55)

while True:
    current_price_lookup = vm.prices.get(user_item, 255)
    ns, vm.dispense, vm.return_change = vm.evaluate_combinational_outputs(user_money, current_price_lookup)

    print(f"\n[CURRENT STATE: {vm.PS}]")
    print(f" -> Registered Balance Tracker: {vm.accumulated_balance}")
    print(f" -> Combinational Output Bus:  [DISPENSE = {vm.dispense}] [RETURN_CHANGE = {vm.return_change}]")
    print("-" * 55)

    if vm.PS == vm.IDLE:
        print("Selection Panel Window:")
        print(f"  Available item codes: 1 to {len(vm.prices)-1}")
        print("  Enter 0 to Shutdown Simulation")
        raw_input = input("Enter selection item digit: ").strip()

        if raw_input == '0':
            print("System software turned off smoothly.")
            break

        if raw_input.isdigit() and int(raw_input) in vm.prices:
            user_item = int(raw_input)
        else:
            print("[!] Invalid Code entered. Item Selection dropped.")
            user_item = 0

        user_money = 0
        vm.tick_clock_edge(ns, user_money, current_price_lookup)
        continue

    elif vm.PS == vm.ADD_MONEY:
        print(f"Active Request Locked: {vm.item_names.get(user_item, 'Unknown')}")
        raw_cash = input("Insert numerical currency value (or 0 to wait): ").strip()
        user_money = int(raw_cash) if raw_cash.isdigit() else 0

        ns, vm.dispense, vm.return_change = vm.evaluate_combinational_outputs(user_money, current_price_lookup)
        vm.tick_clock_edge(ns, user_money, current_price_lookup)
        user_money = 0
        continue

    else:
        # Autonomous state pipelines
        time.sleep(1.5)
        vm.tick_clock_edge(ns, user_money, current_price_lookup)
        user_item = 0


[+] Loaded 100 unique hardware lookup values from the spreadsheet database.

   100+ EXCEL DATABASE PIPELINE RUNTIME ACTIVATED

[CURRENT STATE: IDLE]
 -> Registered Balance Tracker: 0
 -> Combinational Output Bus:  [DISPENSE = 0] [RETURN_CHANGE = 0]
-------------------------------------------------------
Selection Panel Window:
  Available item codes: 1 to 100
  Enter 0 to Shutdown Simulation
Enter selection item digit: 3

[CURRENT STATE: ADD_MONEY]
 -> Registered Balance Tracker: 0
 -> Combinational Output Bus:  [DISPENSE = 0] [RETURN_CHANGE = 0]
-------------------------------------------------------
Active Request Locked: Pepsi (25)
